In [ ]:
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier,RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.metrics import (roc_auc_score,roc_curve,average_precision_score,brier_score_loss,
    accuracy_score,balanced_accuracy_score,precision_score,recall_score,
    confusion_matrix,classification_report,)

warnings.filterwarnings("ignore",category=FutureWarning)
pd.set_option("display.width",160)
pd.set_option("display.max_colwidth",90)
RANDOM_STATE=42
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore",sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore",sparse=False)

def build_preprocessor(cat_cols,num_cols):
    steps=[]
    if cat_cols:
        steps.append(("cat",make_ohe(),list(cat_cols)))
    if num_cols:
        steps.append(("num",StandardScaler(),list(num_cols)))
    return ColumnTransformer(steps)
print("Imports fine.")

Imports fine.


In [ ]:
DATA_PATH=r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\individual_level_london.csv"
df=pd.read_csv(DATA_PATH)
print("Shape:",df.shape)
NEEDED=["survey_year","inactive","age_band","imd_decile","disab3","nssec5","gend3"]
OPTIONAL=["s_bin","borough","LAD24NM","readiness_opportunity","readiness_ability","wt_final","covid_affected","Eth7"]

missing=[c for c in NEEDED if c not in df.columns]
if missing:
    raise KeyError(f"Required columns missing from the file: {missing}")

present_optional=[c for c in OPTIONAL if c in df.columns]
absent_optional=[c for c in OPTIONAL if c not in df.columns]
print("required columns all present.")
print("optional columns present",present_optional)
print("optional columns absent ",absent_optional)

HAS_SBIN="s_bin" in df.columns
HAS_WEIGHTS="wt_final" in df.columns
HAS_READINESS={"readiness_opportunity","readiness_ability"}.issubset(df.columns)

print("rows per survey year:")
print(df["survey_year"].value_counts().sort_index())
print("issing values in the columns")
print(df[NEEDED+present_optional].isna().sum())

Shape: (117679, 13)
Required columns: all present.
Optional columns present: ['s_bin', 'borough', 'readiness_opportunity', 'readiness_ability', 'wt_final', 'covid_affected']
Optional columns absent : ['LAD24NM', 'Eth7']

Rows per survey year:
survey_year
2016-17    19497
2017-18    16200
2018-19    16148
2019-20    16364
2020-21    16340
2021-22    16382
2022-23    16748
Name: count, dtype: int64

Missing values in the columns I model on:
survey_year                  0
inactive                     0
age_band                  1127
imd_decile                   0
disab3                    7944
nssec5                    7270
gend3                      259
s_bin                     1802
borough                      0
readiness_opportunity    46702
readiness_ability        30375
wt_final                     0
covid_affected               0
dtype: int64


In [ ]:
'''checks if missingness is spread or present in one place'''

if "borough" in df.columns:
    complete=df.dropna(subset=NEEDED)
    by_borough=pd.DataFrame({"n_total": df.groupby("borough").size(),
        "n_complete": complete.groupby("borough").size(),}).fillna(0)
    by_borough["n_complete"]=by_borough["n_complete"].astype(int)
    by_borough["pct_retained"]=by_borough["n_complete"] / by_borough["n_total"]
    by_borough=by_borough.sort_values("pct_retained")

    overall_retention=len(complete) / len(df)
    print(f"removng missing demographics{overall_retention:.1%}")
    print("boroughs most affected by the removal")
    print(by_borough.head(8).round(4).to_string())
    spread=by_borough["pct_retained"].max()-by_borough["pct_retained"].min()
    print(f"how it is spread {spread:.1%}")
    if spread > 0.15:
        print("missingness is not evenly distributed")
       
    else:
        print("missingness is not affecting any results")
else:
    print("no borough")

Overall retention after dropping missing demographics: 87.0%

Boroughs with the lowest retention (most affected by missingness):
                        n_total  n_complete  pct_retained
borough                                                  
Kensington and Chelsea     3508        2874        0.8193
Harrow                     3470        2921        0.8418
Camden                     3506        2977        0.8491
Westminster                3477        2956        0.8502
Havering                   3458        2940        0.8502
Bromley                    3435        2923        0.8509
Enfield                    3978        3397        0.8539
Hillingdon                 3455        2951        0.8541

Spread in retention rate across boroughs: 8.6%
Reasonably even. Missingness is unlikely to be distorting any single
borough's residual score disproportionately.


In [ ]:
TRAIN_YEARS=["2016-17","2017-18","2018-19","2019-20","2020-21","2021-22"]
TEST_YEARS=["2022-23"]
EXCLUDE_BOROUGHS=["City of London"]

model_df=df.copy()
model_df["age_band_collapsed"]=(model_df["age_band"].astype(str).replace({"85+": "75+","75-84": "75+","nan": np.nan}))

DEMO_COLS=["age_band_collapsed","imd_decile","disab3","nssec5","gend3"]
CAT_COLS=["age_band_collapsed","disab3","nssec5","gend3"]
NUM_COLS=["imd_decile"]

m1=model_df.dropna(subset=["inactive"]+DEMO_COLS).copy()
m1["inactive"]=m1["inactive"].astype(int)

train_df=m1[m1["survey_year"].isin(TRAIN_YEARS)].copy()
test_df=m1[m1["survey_year"].isin(TEST_YEARS)].copy()

print(f"Complete {len(m1):,} of {len(df):,} rows ({len(m1)/len(df):.1%})")
print(f"Train (2016-17 to 2021-22){len(train_df):,}")
print(f"Test  (2022-23 {len(test_df):,}")
print(f"inactive rate, train: {train_df['inactive'].mean():.4f}")
print(f"inactive rate, test:  {test_df['inactive'].mean():.4f}")

# checking if removal skewed results or not
print("Inactive rate before dropping incomplete cases:", f"{df['inactive'].dropna().astype(int).mean():.4f}")
print("Inactive rate after :", f"{m1['inactive'].mean():.4f}")

Complete cases: 102,401 of 117,679 rows (87.0%)
Train (2016-17 to 2021-22): 88,204
Test  (2022-23):            14,197

Inactive rate, train: 0.2047
Inactive rate, test:  0.2109

Inactive rate before dropping incomplete cases: 0.2261
Inactive rate after : 0.2056


In [ ]:
if HAS_READINESS:
    chk=model_df.copy()
    chk["has_readiness"]=chk["readiness_opportunity"].notna()
    print(f"readiness is there for {chk['has_readiness'].mean():.1%} of rows " f"({chk['has_readiness'].sum():,} of {len(chk):,})")
    print("inactivity rate by whether readiness was asked")
    print(chk.groupby("has_readiness")["inactive"].agg(["mean","size"]).round(4))
    print("readiness coverage by survey year ")
    print(chk.groupby("survey_year")["has_readiness"].mean().round(4))
    for c in ["age_band_collapsed","disab3","nssec5","gend3"]:
        comp=pd.crosstab(chk[c],chk["has_readiness"],normalize="columns").round(4)
        comp.columns=["no_readiness","has_readiness"]
        comp["diff"]=(comp["has_readiness"]-comp["no_readiness"]).round(4)
        biggest=comp["diff"].abs().max()
        print(f"{c}: biggest difference {biggest:.4f}")    
    print("more value means it is present more in a certain place")
else:
    print("No readiness columns")

Readiness present for 60.3% of rows (70,977 of 117,679)

Inactivity rate by whether readiness was asked:
                 mean   size
has_readiness               
False          0.1954  46702
True           0.2464  70977

Readiness coverage by survey year (is it a whole-year design feature?):
survey_year
2016-17    0.2156
2017-18    0.2252
2018-19    0.9627
2019-20    0.4481
2020-21    0.5088
2021-22    0.9654
2022-23    0.9624
Name: has_readiness, dtype: float64



age_band_collapsed: largest composition difference 0.0304


disab3: largest composition difference 0.0275
nssec5: largest composition difference 0.0454


gend3: largest composition difference 0.0068

Small differences mean the readiness subsample is broadly representative.
Large ones, or coverage concentrated in particular years, mean anything built
on readiness describes that subsample and needs saying in the limitations.


In [ ]:
y_test=test_df["inactive"].values
prevalence=y_test.mean()
maj_acc=max(prevalence,1-prevalence)
print(f"most accuracy {maj_acc:.4f}   (AUC is 0.5000 by default")
print(f"inactivity present {prevalence:.4f}   ")
age_pipe=Pipeline([("prep",build_preprocessor(["age_band_collapsed"],[])),("clf",LogisticRegression(max_iter=2000,random_state=RANDOM_STATE)),])
age_pipe.fit(train_df[["age_band_collapsed"]],train_df["inactive"])
p_age=age_pipe.predict_proba(test_df[["age_band_collapsed"]])[:,1]

BASELINE_AUC=roc_auc_score(y_test,p_age)
BASELINE_AP=average_precision_score(y_test,p_age)
print(f"logistic regression using age {BASELINE_AUC:.4f}, " f"average precision {BASELINE_AP:.4f}")

Majority-class accuracy on test: 0.7891   (AUC is 0.5000 by definition)
Test-set prevalence of inactivity: 0.2109   <- floor for average precision



Age-only logistic regression -> test AUC 0.5359, average precision 0.2277

Anything below has to beat these to be worth reporting.


In [ ]:
def pick_threshold(y,p):
    fpr,tpr,thr=roc_curve(y,p)
    return float(thr[np.argmax(tpr-fpr)])
def evaluate(pipe,label,train_d,test_d,feature_cols,target="inactive",
 weight_col=None,store=None,verbose=True):
    X_tr,y_tr=train_d[feature_cols],train_d[target].astype(int).values
    X_te,y_te=test_d[feature_cols],test_d[target].astype(int).values

    fit_kwargs={}
    if weight_col and weight_col in train_d.columns:
        fit_kwargs["clf__sample_weight"]=train_d[weight_col].values

    pipe.fit(X_tr,y_tr,**fit_kwargs)
    p_tr=pipe.predict_proba(X_tr)[:,1]
    p_te=pipe.predict_proba(X_te)[:,1]
    thr=pick_threshold(y_tr,p_tr)
    pred_te=(p_te >= thr).astype(int)
    row={"label": label,
        "n_features": len(feature_cols),"train_auc": roc_auc_score(y_tr,p_tr),"test_auc": roc_auc_score(y_te,p_te),"auc_gap": roc_auc_score(y_tr,p_tr)-roc_auc_score(y_te,p_te),"test_ap": average_precision_score(y_te,p_te),"ap_floor": y_te.mean(),
        "brier": brier_score_loss(y_te,p_te),"thr": thr,"bal_acc": balanced_accuracy_score(y_te,pred_te),
        "recall": recall_score(y_te,pred_te,zero_division=0),"precision": precision_score(y_te,pred_te,zero_division=0),}

    if verbose:
        print(f"{label}")
        print(f"features ({len(feature_cols)}): {feature_cols}")
        print(f"train n={len(X_tr):,}   test n={len(X_te):,}")
        print(f"auc  train {row['train_auc']:.4f} | test {row['test_auc']:.4f} " f"| gap {row['auc_gap']:+.4f}")
        print(f"AP    test  {row['test_ap']:.4f} (floor {row['ap_floor']:.4f})")
        print(f"brier test  {row['brier']:.4f}")
        print(f"At threshold {thr:.3f}: balanced acc {row['bal_acc']:.4f}, "  f"recall {row['recall']:.4f}, precision {row['precision']:.4f}")
 
        print(confusion_matrix(y_te,pred_te))
    if store is not None:
        store.append(row)
    return pipe,p_te,row
results=[]


evaluate() ready.


In [ ]:
# logistic regression using demohgrahics

lr_pipe=Pipeline([ ("prep",build_preprocessor(CAT_COLS,NUM_COLS)),("clf",LogisticRegression(max_iter=2000,C=1.0,random_state=RANDOM_STATE)),])
fitted_lr,p_lr,_=evaluate(lr_pipe,"A: Logistic regression, demographics",train_df,test_df,DEMO_COLS,store=results,)

=== A: Logistic regression, demographics ===
features (5): ['age_band_collapsed', 'imd_decile', 'disab3', 'nssec5', 'gend3']
train n=88,204   test n=14,197
AUC   train 0.6492 | test 0.6580 | gap -0.0088
AP    test  0.3576 (floor 0.2109)
Brier test  0.1561
At threshold 0.210: balanced acc 0.6256, recall 0.5297, precision 0.3370
Confusion matrix (rows actual, cols predicted) [active, inactive]:
[[8083 3120]
 [1408 1586]]



In [ ]:
# imd using 10 categories

train_b=train_df.copy()
test_b=test_df.copy()
for d in (train_b,test_b):
    d["imd_cat"]=d["imd_decile"].astype(int).astype(str)

def add_interactions(tr,te,base_col,cat_col,prefix):
    cats=sorted(tr[cat_col].dropna().unique())
    made=[]
    for cat in cats:
        col=f"{prefix}_{cat}"
        tr[col]=tr[base_col] * (tr[cat_col] == cat).astype(int)
        te[col]=te[base_col] * (te[cat_col] == cat).astype(int)
        made.append(col)
    return made

int_cols=[]
int_cols += add_interactions(train_b,test_b,"imd_decile","disab3","imd_x_disab")
int_cols += add_interactions(train_b,test_b,"imd_decile","age_band_collapsed","imd_x_age")
print(f"Added {len(int_cols)} interaction columns.")

cat_b=["age_band_collapsed","disab3","nssec5","gend3","imd_cat"]
num_b=["imd_decile"]+int_cols
feat_b=cat_b+num_b
lr_rich=Pipeline([("prep",build_preprocessor(cat_b,num_b)),("clf",LogisticRegression(max_iter=3000,C=1.0,random_state=RANDOM_STATE)),])
fitted_lr_rich,p_lr_rich,_=evaluate(lr_rich,"B: Logistic regression, IMD as categorical + interactions",train_b,test_b,feat_b,store=results,)

Added 10 interaction columns.


=== B: Logistic regression, IMD as categorical + interactions ===
features (16): ['age_band_collapsed', 'disab3', 'nssec5', 'gend3', 'imd_cat', 'imd_decile', 'imd_x_disab_Limiting disability', 'imd_x_disab_No disability', 'imd_x_disab_Non-limiting disability', 'imd_x_age_16-24', 'imd_x_age_25-34', 'imd_x_age_35-44', 'imd_x_age_45-54', 'imd_x_age_55-64', 'imd_x_age_65-74', 'imd_x_age_75+']
train n=88,204   test n=14,197
AUC   train 0.6509 | test 0.6593 | gap -0.0084
AP    test  0.3576 (floor 0.2109)
Brier test  0.1560
At threshold 0.214: balanced acc 0.6250, recall 0.5147, precision 0.3420
Confusion matrix (rows actual, cols predicted) [active, inactive]:
[[8238 2965]
 [1453 1541]]



In [ ]:
# gradient boost

gb_loose=Pipeline([("prep",build_preprocessor(CAT_COLS,NUM_COLS)),("clf",HistGradientBoostingClassifier(max_iter=500,learning_rate=0.1,max_leaf_nodes=63,min_samples_leaf=5,l2_regularization=0.0,early_stopping=False,random_state=RANDOM_STATE)),])
fitted_gb_loose,p_gb_loose,_=evaluate(gb_loose,"no constraints for gradient boosting",train_df,test_df,DEMO_COLS,store=results,)

gb_tight=Pipeline([("prep",build_preprocessor(CAT_COLS,NUM_COLS)),("clf",HistGradientBoostingClassifier(max_iter=400,learning_rate=0.05,max_leaf_nodes=15,min_samples_leaf=200,l2_regularization=1.0,early_stopping=True,validation_fraction=0.15,n_iter_no_change=25,random_state=RANDOM_STATE)),])
fitted_gb,p_gb,_=evaluate(gb_tight,"gradient boosting stop early and regularised",train_df,test_df,DEMO_COLS,store=results,)
n_used=fitted_gb.named_steps["clf"].n_iter_
print(f"stopped {n_used} ")

=== C1: Gradient boosting, deliberately unconstrained ===
features (5): ['age_band_collapsed', 'imd_decile', 'disab3', 'nssec5', 'gend3']
train n=88,204   test n=14,197
AUC   train 0.6798 | test 0.6382 | gap +0.0417
AP    test  0.3352 (floor 0.2109)
Brier test  0.1591
At threshold 0.204: balanced acc 0.6099, recall 0.4830, precision 0.3290
Confusion matrix (rows actual, cols predicted) [active, inactive]:
[[8254 2949]
 [1548 1446]]



=== C2: Gradient boosting, regularised + early stopping ===
features (5): ['age_band_collapsed', 'imd_decile', 'disab3', 'nssec5', 'gend3']
train n=88,204   test n=14,197
AUC   train 0.6579 | test 0.6617 | gap -0.0039
AP    test  0.3628 (floor 0.2109)
Brier test  0.1556
At threshold 0.204: balanced acc 0.6261, recall 0.5190, precision 0.3421
Confusion matrix (rows actual, cols predicted) [active, inactive]:
[[8214 2989]
 [1440 1554]]

C2 stopped early at 137 boosting iterations out of a possible 400.


In [ ]:
# model with baseline

summary=pd.DataFrame(results)
summary.insert(1,"beats_age_only",(summary["test_auc"]-BASELINE_AUC).round(4))
show=summary[["label","train_auc","test_auc","auc_gap","beats_age_only","test_ap","ap_floor","brier","bal_acc"]]
print(f"Reference: age-only baseline test AUC = {BASELINE_AUC:.4f}, " f"majority-class AUC = 0.5000")
print(show.round(4).to_string(index=False))
print("large positive gap means overfiited model")

Reference: age-only baseline test AUC = 0.5359, majority-class AUC = 0.5000

                                                    label  train_auc  test_auc  auc_gap  beats_age_only  test_ap  ap_floor  brier  bal_acc
                     A: Logistic regression, demographics     0.6492    0.6580  -0.0088          0.1221   0.3576    0.2109 0.1561   0.6256
B: Logistic regression, IMD as categorical + interactions     0.6509    0.6593  -0.0084          0.1234   0.3576    0.2109 0.1560   0.6250
        C1: Gradient boosting, deliberately unconstrained     0.6798    0.6382   0.0417          0.1022   0.3352    0.2109 0.1591   0.6099
      C2: Gradient boosting, regularised + early stopping     0.6579    0.6617  -0.0039          0.1258   0.3628    0.2109 0.1556   0.6261

Reading the gap column: a small positive gap is normal and healthy. A large one
means the model memorised the training years. A gap near zero with a low test AUC
means the model is underfitting, or there just isn't much signal 

In [ ]:
def make_gb():
    return Pipeline([("prep",build_preprocessor(CAT_COLS,NUM_COLS)),("clf",HistGradientBoostingClassifier(max_iter=400,learning_rate=0.05,max_leaf_nodes=15,min_samples_leaf=200,l2_regularization=1.0,early_stopping=True,validation_fraction=0.15,n_iter_no_change=25,random_state=RANDOM_STATE)),])
def make_lr():
    return Pipeline([("prep",build_preprocessor(CAT_COLS,NUM_COLS)),("clf",LogisticRegression(max_iter=2000,random_state=RANDOM_STATE)),])
def rolling_origin(make_pipe,d,feature_cols,target="inactive",min_train_years=2):
    years=sorted(d["survey_year"].dropna().unique())
    rows=[]
    for i in range(min_train_years,len(years)):
        tr=d[d["survey_year"].isin(years[:i])]
        te=d[d["survey_year"] == years[i]]
        if len(te) < 100 or te[target].nunique() < 2 or tr[target].nunique() < 2:
            continue
        pipe=make_pipe()
        pipe.fit(tr[feature_cols],tr[target].astype(int))
        p=pipe.predict_proba(te[feature_cols])[:,1]
        y=te[target].astype(int).values
        rows.append({"trained_through": years[i-1],
            "tested_on": years[i],"n_train": len(tr),
            "n_test": len(te),"test_auc": roc_auc_score(y,p),
            "test_ap": average_precision_score(y,p),"prevalence": y.mean(),})
    return pd.DataFrame(rows)

roll_gb=rolling_origin(make_gb,m1,DEMO_COLS)
roll_lr=rolling_origin(make_lr,m1,DEMO_COLS)

print(" gradient boosting")
print(roll_gb.round(4).to_string(index=False))
print("logistic regression")
print(roll_lr.round(4).to_string(index=False))
print(f"GB  test AUC across folds: mean {roll_gb['test_auc'].mean():.4f}, " f"sd {roll_gb['test_auc'].std():.4f}, " f"range {roll_gb['test_auc'].min():.4f} to {roll_gb['test_auc'].max():.4f}")
print(f"LR  test AUC across folds: mean {roll_lr['test_auc'].mean():.4f}, "f"sd {roll_lr['test_auc'].std():.4f}, "f"range {roll_lr['test_auc'].min():.4f} to {roll_lr['test_auc'].max():.4f}")

Rolling-origin, gradient boosting:
trained_through tested_on  n_train  n_test  test_auc  test_ap  prevalence
        2017-18   2018-19    32109   13916    0.6409   0.2988      0.1840
        2018-19   2019-20    46025   14148    0.6565   0.3497      0.2103
        2019-20   2020-21    60173   14035    0.6417   0.3491      0.2208
        2020-21   2021-22    74208   13996    0.6513   0.3429      0.2091
        2021-22   2022-23    88204   14197    0.6617   0.3628      0.2109

Rolling-origin, logistic regression:
trained_through tested_on  n_train  n_test  test_auc  test_ap  prevalence
        2017-18   2018-19    32109   13916    0.6426   0.3013      0.1840
        2018-19   2019-20    46025   14148    0.6530   0.3503      0.2103
        2019-20   2020-21    60173   14035    0.6396   0.3429      0.2208
        2020-21   2021-22    74208   13996    0.6467   0.3385      0.2091
        2021-22   2022-23    88204   14197    0.6580   0.3576      0.2109

GB  test AUC across folds: mean 0.6504

In [ ]:
# confidence intervals are bootsrapped
def bootstrap_auc_ci(y,p,n_boot=1000,seed=RANDOM_STATE):
    rng=np.random.default_rng(seed)
    y,p=np.asarray(y),np.asarray(p)
    out=[]
    for _ in range(n_boot):
        idx=rng.integers(0,len(y),len(y))
        if len(np.unique(y[idx])) < 2:
            continue
        out.append(roc_auc_score(y[idx],p[idx]))
    out=np.array(out)
    return out.mean(),np.percentile(out,2.5),np.percentile(out,97.5)

def bootstrap_auc_delta(y,p_a,p_b,n_boot=1000,seed=RANDOM_STATE):
    rng=np.random.default_rng(seed)
    y,p_a,p_b=np.asarray(y),np.asarray(p_a),np.asarray(p_b)
    out=[]
    for _ in range(n_boot):
        idx=rng.integers(0,len(y),len(y))
        if len(np.unique(y[idx])) < 2:
            continue
        out.append(roc_auc_score(y[idx],p_b[idx])-roc_auc_score(y[idx],p_a[idx]))
    out=np.array(out)
    return out.mean(),np.percentile(out,2.5),np.percentile(out,97.5)

for name,p in [("  logistic regression",p_lr),(" lr + interactions",p_lr_rich),(" gradient boosting",p_gb)]:
    m,lo,hi=bootstrap_auc_ci(y_test,p)
    print(f"{name:24s} test AUC {m:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")
m,lo,hi=bootstrap_auc_delta(y_test,p_lr,p_gb)
print(f"gb minus lr {m:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]")


A  logistic regression   test AUC 0.6581  95% CI [0.6470, 0.6697]


B  LR + interactions     test AUC 0.6594  95% CI [0.6484, 0.6713]


C2 gradient boosting     test AUC 0.6619  95% CI [0.6508, 0.6735]



GB minus LR: +0.0038  95% CI [+0.0005, +0.0072]
If that interval spans zero, the two models are not distinguishable on this test set,
and I should report the simpler, interpretable one as the main model.


In [ ]:
fractions=[0.05,0.1,0.2,0.4,0.6,0.8,1.0]
curve=[]
for frac in fractions:
    sub=train_df.sample(frac=frac,random_state=RANDOM_STATE)
    if sub["inactive"].nunique() < 2:
        continue
    pipe=make_gb()
    pipe.fit(sub[DEMO_COLS],sub["inactive"])
    p_sub_tr=pipe.predict_proba(sub[DEMO_COLS])[:,1]
    p_sub_te=pipe.predict_proba(test_df[DEMO_COLS])[:,1]
    curve.append({"frac": frac,
        "n_train": len(sub),"train_auc": roc_auc_score(sub["inactive"],p_sub_tr),"test_auc": roc_auc_score(y_test,p_sub_te),})

curve_df=pd.DataFrame(curve)
curve_df["gap"]=curve_df["train_auc"]-curve_df["test_auc"]
print(curve_df.round(4).to_string(index=False))
last_three=curve_df["test_auc"].tail(3).values
print(f" test AUC {last_three[-1] - last_three[0]:+.4f}")


 frac  n_train  train_auc  test_auc     gap
 0.05     4410     0.6661    0.6481  0.0180
 0.10     8820     0.6753    0.6562  0.0191
 0.20    17641     0.6695    0.6593  0.0102
 0.40    35282     0.6590    0.6575  0.0015
 0.60    52922     0.6578    0.6612 -0.0034
 0.80    70563     0.6591    0.6616 -0.0025
 1.00    88204     0.6584    0.6624 -0.0040

Test AUC change over the last three points: +0.0012
Near zero means the curve has flattened and more rows would not help.


In [ ]:
# calibaration
frac_pos,mean_pred=calibration_curve(y_test,p_gb,n_bins=10,strategy="quantile")
cal=pd.DataFrame({"mean_predicted_risk": mean_pred,"actual_inactive_rate": frac_pos,"difference": frac_pos-mean_pred,})
print("calibration, gradient boosting model")
print(cal.round(4).to_string(index=False))
print(f"brier score: {brier_score_loss(y_test, p_gb):.4f}")
print(f"{brier_score_loss(y_test, np.full_like(p_gb, prevalence)):.4f}")
print(f"max cal error {cal['difference'].abs().max():.4f}")
#refit model
lr_bal=Pipeline([("prep",build_preprocessor(CAT_COLS,NUM_COLS)),("clf",LogisticRegression(max_iter=2000,class_weight="balanced",random_state=RANDOM_STATE)),])
lr_bal.fit(train_df[DEMO_COLS],train_df["inactive"])
p_bal=lr_bal.predict_proba(test_df[DEMO_COLS])[:,1]
print(f"  AUC   {roc_auc_score(y_test, p_bal):.4f}  (basically unchanged)")
print(f"  brier {brier_score_loss(y_test, p_bal):.4f}  (much worse)")
print(f"  mean predicted risk {p_bal.mean():.4f} vs actual rate {prevalence:.4f}")


Calibration, gradient boosting model, test year:
 mean_predicted_risk  actual_inactive_rate  difference
              0.1029                0.1241      0.0212
              0.1223                0.1218     -0.0006
              0.1362                0.1312     -0.0050
              0.1473                0.1517      0.0044
              0.1575                0.1548     -0.0027
              0.1702                0.1694     -0.0008
              0.1955                0.2109      0.0154
              0.2330                0.2568      0.0238
              0.2873                0.3243      0.0371
              0.4115                0.4658      0.0544

Brier score: 0.1556
Brier for always predicting the base rate: 0.1664
Max absolute calibration error: 0.0544

For contrast, the same model refit with class_weight='balanced':


  AUC   0.6582  (basically unchanged)
  Brier 0.2223  (much worse)
  mean predicted risk 0.4671 vs actual rate 0.2109
That is why class_weight is off in the reported models. It shifts the threshold,
not the ranking, and it destroys the probabilities in the process.


In [ ]:
# permutation

perm=permutation_importance(fitted_gb,test_df[DEMO_COLS],test_df["inactive"],scoring="roc_auc",n_repeats=10,random_state=RANDOM_STATE,n_jobs=1,)
imp=(pd.DataFrame({"feature": DEMO_COLS,"auc_drop": perm.importances_mean,"sd": perm.importances_std,}).sort_values("auc_drop",ascending=False).reset_index(drop=True))
print("permutation importance ")
print(imp.round(4).to_string(index=False))

Permutation importance (drop in test AUC when the column is shuffled):
           feature  auc_drop     sd
            nssec5    0.0724 0.0031
            disab3    0.0310 0.0017
        imd_decile    0.0191 0.0014
age_band_collapsed    0.0110 0.0013
             gend3    0.0036 0.0009

Anything at or below zero contributes nothing out of sample, whatever the
in-sample importance says.


In [ ]:
# checking if s bin improves anything

if HAS_SBIN:
    m1s=m1.dropna(subset=["s_bin"]).copy()
    tr_s=m1s[m1s["survey_year"].isin(TRAIN_YEARS)].copy()
    te_s=m1s[m1s["survey_year"].isin(TEST_YEARS)].copy()
    y_s=te_s["inactive"].astype(int).values
    print(f"rows with s_bin {len(m1s):,} of {len(m1):,} "f"({len(m1s)/len(m1):.1%})")
    print(f"Train {len(tr_s):,} | Test {len(te_s):,}")
    sbin_results=[]
    _,p_nos,_=evaluate( make_gb(),"D1: GB, demographics only (s_bin-complete rows)",tr_s,te_s,DEMO_COLS,store=sbin_results,)
    _,p_yess,_=evaluate(Pipeline([("prep",build_preprocessor(CAT_COLS+["s_bin"],NUM_COLS)),("clf",HistGradientBoostingClassifier(max_iter=400,learning_rate=0.05,max_leaf_nodes=15,min_samples_leaf=200,l2_regularization=1.0,early_stopping=True,validation_fraction=0.15,n_iter_no_change=25,random_state=RANDOM_STATE)),]),"D2: GB, demographics + s_bin",tr_s,te_s,DEMO_COLS+["s_bin"],store=sbin_results,)

    d_mean,d_lo,d_hi=bootstrap_auc_delta(y_s,p_nos,p_yess)
    print(f"AUC change after s bin {d_mean:+.4f}  95% CI [{d_lo:+.4f}, {d_hi:+.4f}]")
    if d_lo <= 0 <= d_hi:
        print(" sbin does not improve individual level")
    else:
        print(" s_bin does shift prediction.")
    
    results.extend(sbin_results)
else:
    print("ignore")

Rows with s_bin present: 100,839 of 102,401 (98.5%)
Train 86,844 | Test 13,995



=== D1: GB, demographics only (s_bin-complete rows) ===
features (5): ['age_band_collapsed', 'imd_decile', 'disab3', 'nssec5', 'gend3']
train n=86,844   test n=13,995
AUC   train 0.6589 | test 0.6703 | gap -0.0114
AP    test  0.3646 (floor 0.2079)
Brier test  0.1532
At threshold 0.211: balanced acc 0.6312, recall 0.5199, precision 0.3465
Confusion matrix (rows actual, cols predicted) [active, inactive]:
[[8231 2854]
 [1397 1513]]



=== D2: GB, demographics + s_bin ===
features (6): ['age_band_collapsed', 'imd_decile', 'disab3', 'nssec5', 'gend3', 's_bin']
train n=86,844   test n=13,995
AUC   train 0.6629 | test 0.6706 | gap -0.0076
AP    test  0.3654 (floor 0.2079)
Brier test  0.1532
At threshold 0.207: balanced acc 0.6315, recall 0.5440, precision 0.3370
Confusion matrix (rows actual, cols predicted) [active, inactive]:
[[7971 3114]
 [1327 1583]]



AUC change from adding s_bin: +0.0003  95% CI [-0.0022, +0.0027]
Interval spans zero. On this evidence s_bin does not improve individual-level
prediction, which lines up with the ICC of roughly 1% from the multilevel model.


In [ ]:
# adding borough and covid data to see if it makes chnage

place_cols,place_cats=list(DEMO_COLS),list(CAT_COLS)
if "borough" in m1.columns:
    place_cols,place_cats=place_cols+["borough"],place_cats+["borough"]
if "covid_affected" in m1.columns:
    m1["covid_flag"]=m1["covid_affected"].astype(str)
    train_df["covid_flag"]=train_df["covid_affected"].astype(str)
    test_df["covid_flag"]=test_df["covid_affected"].astype(str)
    place_cols,place_cats=place_cols+["covid_flag"],place_cats+["covid_flag"]

if len(place_cols) > len(DEMO_COLS):
    print(f"extra columns  {[c for c in place_cols if c not in DEMO_COLS]}")
    if "borough" in m1.columns:
        print(f"Distinct boroughs: {m1['borough'].nunique()}")

    place_store=[]
    _,p_base,_=evaluate(make_gb(),"G1: GB, demographics only (reference)",train_df,test_df,DEMO_COLS,store=place_store,verbose=False,)
    _,p_place,_=evaluate(Pipeline([("prep",build_preprocessor(place_cats,NUM_COLS)),("clf",HistGradientBoostingClassifier(max_iter=400,learning_rate=0.05,max_leaf_nodes=15,min_samples_leaf=200,l2_regularization=1.0,early_stopping=True,validation_fraction=0.15,n_iter_no_change=25,random_state=RANDOM_STATE)),]),"G2: GB, demographics + borough/covid",train_df,test_df,place_cols,store=place_store,verbose=False,)
    print(pd.DataFrame(place_store)[["label","n_features","train_auc","test_auc","auc_gap"]].round(4).to_string(index=False))
    
    d_mean,d_lo,d_hi=bootstrap_auc_delta(y_test,p_base,p_place)
    print(f"AUC change  {d_mean:+.4f}  95% CI [{d_lo:+.4f}, {d_hi:+.4f}]")
    if d_lo <= 0 <= d_hi:
        print("doesn't add meaningful information")
    elif d_hi < 0:
        print("negatively significant")
    else:
        print("Significantly positive")
    results.extend(place_store)
else:
    print("ignore")

Extra columns being tested: ['borough', 'covid_flag']
Distinct boroughs: 33



                                label  n_features  train_auc  test_auc  auc_gap
G1: GB, demographics only (reference)           5     0.6579    0.6617  -0.0039
 G2: GB, demographics + borough/covid           7     0.6849    0.6690   0.0159



AUC change from adding place: +0.0070  95% CI [+0.0010, +0.0132]
Significantly positive. Report the effect size, not just the sign, and
check it isn't the boosting model exploiting 33 dummies on a large training
set. Compare the auc_gap column between G1 and G2 before believing it.


In [ ]:
'''which decile are more at risk'''
risk=pd.DataFrame({"p": p_gb,"inactive": y_test})
risk["decile"]=pd.qcut(risk["p"].rank(method="first"),10,labels=range(1,11)).astype(int)

lift=(risk.groupby("decile").agg(n=("inactive","size"),mean_predicted=("p","mean"),actual_rate=("inactive","mean"),n_inactive=("inactive","sum")).sort_index(ascending=False))
lift["lift_vs_base"]=lift["actual_rate"] / prevalence
lift["cum_pct_population"]=(lift["n"].cumsum() / lift["n"].sum())
lift["cum_pct_inactive_captured"]=(lift["n_inactive"].cumsum() / lift["n_inactive"].sum())

print(f"inactivity rate in test year {prevalence:.4f}")

print(lift.round(4).to_string())
top2=lift["cum_pct_inactive_captured"].iloc[1]
top3=lift["cum_pct_inactive_captured"].iloc[2]
print(f"top 20% {top2:.1%} ")
print(f"top 30% {top3:.1%}.")


Base inactivity rate in test year: 0.2109
Deciles ordered highest predicted risk first.

           n  mean_predicted  actual_rate  n_inactive  lift_vs_base  cum_pct_population  cum_pct_inactive_captured
decile                                                                                                            
10      1420          0.4114       0.4662         662        2.2106                 0.1                     0.2211
9       1420          0.2869       0.3232         459        1.5327                 0.2                     0.3744
8       1419          0.2328       0.2565         364        1.2164                 0.3                     0.4960
7       1420          0.1954       0.2106         299        0.9985                 0.4                     0.5959
6       1419          0.1701       0.1691         240        0.8020                 0.5                     0.6760
5       1420          0.1573       0.1521         216        0.7213                 0.6                   

In [ ]:
#old code that had leakage

SOLO_CERTAIN,SOLO_SUSPECT,GROUP_FLAG=0.95,0.80,0.98
def leakage_screen(d,feature_cols,target):
    y=d[target].astype(int).values
    rows=[]
    for c in feature_cols:
        s=d[c]
        if pd.api.types.is_numeric_dtype(s):
            score=s.fillna(s.median()).astype(float).values
        else:
            rate=d.groupby(c)[target].mean()
            score=s.map(rate).fillna(y.mean()).astype(float).values
        if len(np.unique(y)) < 2 or np.all(score == score[0]):
            continue
        a=max(roc_auc_score(y,score),1-roc_auc_score(y,score))
        verdict=("leakage" if a >= SOLO_CERTAIN
                   else "check" if a >= SOLO_SUSPECT else "ok")
        rows.append({"feature": c,"solo_auc": a,"verdict": verdict})
    return (pd.DataFrame(rows).sort_values("solo_auc",ascending=False).reset_index(drop=True))
def group_leakage_check(tr,te,feature_cols,target,cat_cols,num_cols):
    pipe=Pipeline([("prep",build_preprocessor(cat_cols,num_cols)),("clf",HistGradientBoostingClassifier(max_iter=150,max_leaf_nodes=31,random_state=RANDOM_STATE)),])
    pipe.fit(tr[feature_cols],tr[target].astype(int))
    auc=roc_auc_score(te[target].astype(int),pipe.predict_proba(te[feature_cols])[:,1])
    verdict=("leakage" if auc >= GROUP_FLAG
               else "check" if auc >= 0.90 else "ok")
    print(f"check {auc:.4f}   {verdict}")
    return auc,verdict

print(leakage_screen(train_df,DEMO_COLS,"inactive").round(4).to_string(index=False))
group_leakage_check(train_df,test_df,DEMO_COLS,"inactive",CAT_COLS,NUM_COLS)

Screen on the inactivity feature set (this is the set I actually report):


           feature  solo_auc verdict
            nssec5    0.6106      ok
        imd_decile    0.5623      ok
            disab3    0.5610      ok
age_band_collapsed    0.5469      ok
             gend3    0.5127      ok


Group check: joint test AUC 0.6623  ->  ok

Nothing flagged on either part. That is what a clean feature set looks like.


In [ ]:
if HAS_READINESS:
    bar=model_df.dropna(subset=["readiness_opportunity","readiness_ability"]+DEMO_COLS).copy()
    bar_tr=bar[bar["survey_year"].isin(TRAIN_YEARS)].copy()
    bar_te=bar[bar["survey_year"].isin(TEST_YEARS)].copy()
    avg_opp=bar_tr["readiness_opportunity"].mean()
    avg_abi=bar_tr["readiness_ability"].mean()

    for d in (bar_tr,bar_te):
        d["dominant_barrier"]=np.where((avg_opp-d["readiness_opportunity"]) > (avg_abi-d["readiness_ability"]),"opportunity","ability")
        d["target"]=(d["dominant_barrier"]=="opportunity").astype(int)

    leaky_feats=DEMO_COLS+["readiness_opportunity","readiness_ability"]
    leaky_num=NUM_COLS+["readiness_opportunity","readiness_ability"]

    print(leakage_screen(bar_tr,leaky_feats,"target").round(4).to_string(index=False))
    group_leakage_check(bar_tr,bar_te,leaky_feats,"target",CAT_COLS,leaky_num)

    leak_store=[]
    evaluate(Pipeline([("prep",build_preprocessor(CAT_COLS,NUM_COLS+["readiness_opportunity","readiness_ability"])),("clf",LogisticRegression(max_iter=2000,random_state=RANDOM_STATE)),]),bar_tr,bar_te,leaky_feats,target="target",store=leak_store,)
   
else:
    print("ignore")

Leakage screen BEFORE fitting anything.
Part 1, solo:


              feature  solo_auc           verdict
readiness_opportunity    0.8316 suspicious, check
    readiness_ability    0.6237                ok
   age_band_collapsed    0.5923                ok
               nssec5    0.5316                ok
           imd_decile    0.5270                ok
               disab3    0.5254                ok
                gend3    0.5150                ok

Part 2, group:


Group check: joint test AUC 1.0000  ->  LEAKAGE, near-certain



=== LEAKY (not reported): barrier type from the columns it was built from ===
features (7): ['age_band_collapsed', 'imd_decile', 'disab3', 'nssec5', 'gend3', 'readiness_opportunity', 'readiness_ability']
train n=48,176   test n=13,815
AUC   train 1.0000 | test 1.0000 | gap +0.0000
AP    test  1.0000 (floor 0.2021)
Brier test  0.0000
At threshold 0.994: balanced acc 1.0000, recall 1.0000, precision 1.0000
Confusion matrix (rows actual, cols predicted) [active, inactive]:
[[11023     0]
 [    0  2792]]

Note what each part of the screen did. The solo scores are high but not
damning, because the label depends on the DIFFERENCE between the two
columns and neither one carries it alone. The group check is the part that
catches it outright. That is exactly why the screen has two parts.


In [ ]:
# trying to improve the keakage model

if HAS_READINESS:
    opp=model_df.dropna(subset=["readiness_opportunity"]+DEMO_COLS).copy()
    opp_tr_raw=opp[opp["survey_year"].isin(TRAIN_YEARS)]
    dist=opp_tr_raw["readiness_opportunity"].value_counts(normalize=True).sort_index()
    print(dist.round(4).to_string())

    candidates=sorted(opp_tr_raw["readiness_opportunity"].dropna().unique())[:-1]
    options=[(c,(opp_tr_raw["readiness_opportunity"] <= c).mean()) for c in candidates]
    for c,rate in options:
        print(f"  <= {c:g}  ->  {rate:.4f}")
    CUT=min(options,key=lambda t: abs(t[1]-1/3))[0]
    chosen_rate=dict(options)[CUT]
    
    print(f"readiness_opportunity <= {CUT:g} uses {chosen_rate:.4f}")
    if not 0.15 <= chosen_rate <= 0.55:
        print("just trying")
      
    opp["low_opportunity"]=(opp["readiness_opportunity"] <= CUT).astype(int)
    opp_tr=opp[opp["survey_year"].isin(TRAIN_YEARS)].copy()
    opp_te=opp[opp["survey_year"].isin(TEST_YEARS)].copy()

    print(f"train {len(opp_tr):,} | Test {len(opp_te):,}")
    print(f"positive rate, train {opp_tr['low_opportunity'].mean():.4f} | "
f"test {opp_te['low_opportunity'].mean():.4f}")

    feats_opp=DEMO_COLS+(["s_bin"] if HAS_SBIN else [])
    cats_opp=CAT_COLS+(["s_bin"] if HAS_SBIN else [])
    opp_tr=opp_tr.dropna(subset=feats_opp)
    opp_te=opp_te.dropna(subset=feats_opp)

    print(leakage_screen(opp_tr,feats_opp,"low_opportunity").round(4).to_string(index=False))
    group_leakage_check(opp_tr,opp_te,feats_opp,"low_opportunity",cats_opp,NUM_COLS)

    opp_store=[]
    evaluate(Pipeline([("prep",build_preprocessor(cats_opp,NUM_COLS)),("clf",LogisticRegression(max_iter=2000,random_state=RANDOM_STATE)),]),"E1: LR, low opportunity readiness from demographics",opp_tr,opp_te,feats_opp,target="low_opportunity",store=opp_store,)
    _,p_opp,_=evaluate(Pipeline([("prep",build_preprocessor(cats_opp,NUM_COLS)),("clf",HistGradientBoostingClassifier(max_iter=400,learning_rate=0.05,max_leaf_nodes=15,min_samples_leaf=200,l2_regularization=1.0,early_stopping=True,validation_fraction=0.15,n_iter_no_change=25,random_state=RANDOM_STATE)),]),"E2: GB, low opportunity readiness from demographics",opp_tr,opp_te,feats_opp,target="low_opportunity",store=opp_store,)

    y_opp=opp_te["low_opportunity"].values
    m,lo,hi=bootstrap_auc_ci(y_opp,p_opp)
    print(f" auc {m:.4f}, 95% ci [{lo:.4f}, {hi:.4f}]")
    results.extend(opp_store)
else:
    print("ignore")

Distribution of readiness_opportunity (train years):
readiness_opportunity
1.0    0.0202
2.0    0.0647
3.0    0.1043
4.0    0.4588
5.0    0.3520

Candidate cut points and the positive rate each would give:
  <= 1  ->  0.0202
  <= 2  ->  0.0849
  <= 3  ->  0.1892
  <= 4  ->  0.6480

Chosen cut: readiness_opportunity <= 3, positive rate 0.1892


Train 48,462 | Test 13,892
Positive rate, train 0.1892 | test 0.1784

Leakage screen first, both parts:


           feature  solo_auc verdict
            disab3    0.5883      ok
        imd_decile    0.5539      ok
            nssec5    0.5519      ok
             gend3    0.5316      ok
age_band_collapsed    0.5222      ok
             s_bin    0.5098      ok


Group check: joint test AUC 0.6583  ->  ok



=== E1: LR, low opportunity readiness from demographics ===
features (6): ['age_band_collapsed', 'imd_decile', 'disab3', 'nssec5', 'gend3', 's_bin']
train n=47,581   test n=13,690
AUC   train 0.6432 | test 0.6560 | gap -0.0128
AP    test  0.3143 (floor 0.1798)
Brier test  0.1386
At threshold 0.188: balanced acc 0.6130, recall 0.4464, precision 0.3076
Confusion matrix (rows actual, cols predicted) [active, inactive]:
[[8754 2474]
 [1363 1099]]



=== E2: GB, low opportunity readiness from demographics ===
features (6): ['age_band_collapsed', 'imd_decile', 'disab3', 'nssec5', 'gend3', 's_bin']
train n=47,581   test n=13,690
AUC   train 0.6550 | test 0.6589 | gap -0.0039
AP    test  0.3261 (floor 0.1798)
Brier test  0.1381
At threshold 0.187: balanced acc 0.6157, recall 0.4976, precision 0.2908
Confusion matrix (rows actual, cols predicted) [active, inactive]:
[[8240 2988]
 [1237 1225]]



E2 test AUC 0.6592, 95% CI [0.6465, 0.6714]


In [ ]:
#sensitivity check
if HAS_WEIGHTS:
    tr_w=train_df.dropna(subset=["wt_final"])
    te_w=test_df.dropna(subset=["wt_final"])
    w_store=[]
    evaluate(make_lr(),"F1: LR, unweighted (reference)",tr_w,te_w,DEMO_COLS,store=w_store,verbose=False)
    evaluate(make_lr(),"F2: LR, survey-weighted fit",tr_w,te_w,DEMO_COLS,weight_col="wt_final",store=w_store,verbose=False)
    wdf=pd.DataFrame(w_store)[["label","train_auc","test_auc","brier"]]
    print(wdf.round(4).to_string(index=False))
    results.extend(w_store)
else:
    print("ignore")

                         label  train_auc  test_auc  brier
F1: LR, unweighted (reference)     0.6492    0.6580 0.1561
   F2: LR, survey-weighted fit     0.6480    0.6578 0.1567

If these are close, the unweighted model is fine to report for prediction,
and the weights only matter for the descriptive prevalence figures elsewhere.


In [ ]:
#saving table

final=pd.DataFrame(results)
final["beats_age_only"]=(final["test_auc"]-BASELINE_AUC).round(4)
cols=["label","n_features","train_auc","test_auc","auc_gap","beats_age_only","test_ap","ap_floor","brier","bal_acc","recall","precision"]
final=final[cols].round(4)

print(f"using age only {BASELINE_AUC:.4f} ")
print(final.to_string(index=False))

OUT=r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\model_results_v2.csv"
final.to_csv(OUT,index=False)
print(f" {OUT}")

Age-only baseline test AUC: 0.5359 | majority-class AUC: 0.5000

                                                    label  n_features  train_auc  test_auc  auc_gap  beats_age_only  test_ap  ap_floor  brier  bal_acc  recall  precision
                     A: Logistic regression, demographics           5     0.6492    0.6580  -0.0088          0.1221   0.3576    0.2109 0.1561   0.6256  0.5297     0.3370
B: Logistic regression, IMD as categorical + interactions          16     0.6509    0.6593  -0.0084          0.1234   0.3576    0.2109 0.1560   0.6250  0.5147     0.3420
        C1: Gradient boosting, deliberately unconstrained           5     0.6798    0.6382   0.0417          0.1022   0.3352    0.2109 0.1591   0.6099  0.4830     0.3290
      C2: Gradient boosting, regularised + early stopping           5     0.6579    0.6617  -0.0039          0.1258   0.3628    0.2109 0.1556   0.6261  0.5190     0.3421
          D1: GB, demographics only (s_bin-complete rows)           5     0.6589    0

In [ ]:
# checks individual risk minus inactivity

if "borough" in test_df.columns:
    bor=test_df[["borough","inactive"]].copy()
    bor["expected"]=p_gb
    if HAS_WEIGHTS:
        bor["w"]=test_df["wt_final"].values
    else:
        bor["w"]=1.0

    n_before=bor["borough"].nunique()
    bor=bor[~bor["borough"].isin(EXCLUDE_BOROUGHS)].copy()
    print(f" {EXCLUDE_BOROUGHS}: {n_before} boroughs  {bor['borough'].nunique()}")

    def wmean(s,w):
        return np.average(s,weights=w) if w.sum() > 0 else np.nan

    rows=[]
    for b,g in bor.groupby("borough"):
        obs=wmean(g["inactive"].values,g["w"].values)
        exp=wmean(g["expected"].values,g["w"].values)
        
        n_eff=(g["w"].sum()**2)/(g["w"]**2).sum()  
        se=np.sqrt(max(obs*(1-obs),1e-9)/max(n_eff,1))
        rows.append({"borough": b,"n":len(g),"n_effective": n_eff,"observed": obs,"expected": exp,"residual": obs-exp,"se": se})
    bres=pd.DataFrame(rows)
#bayes shrinkage
    tau2=max(0.0,bres["residual"].var(ddof=1)-(bres["se"] ** 2).mean())
    bres["shrinkage_weight"]=tau2 / (tau2+bres["se"] ** 2)
    bres["residual_shrunk"]=bres["residual"] * bres["shrinkage_weight"]
    bres["ci_lo"]=bres["residual"]-1.96 * bres["se"]
    bres["ci_hi"]=bres["residual"]+1.96 * bres["se"]
    bres["excess_significant"]=(bres["ci_lo"] > 0) | (bres["ci_hi"] < 0)
    bres=bres.sort_values("residual_shrunk",ascending=False).reset_index(drop=True)
    print(f" {tau2:.6f}")
    print(f"sd {np.sqrt(tau2):.4f}")
    print(f"shrinkage {bres['shrinkage_weight'].mean():.3f} ")
    print(f"shrinkage differs" f"{int(bres['excess_significant'].sum())} of {len(bres)}")
    print(bres[["borough","n","observed","expected","residual","residual_shrunk","excess_significant"]].round(4).to_string(index=False))

    if "borough" in df.columns:
        low_retention=set(by_borough.sort_values("pct_retained").head(8).index)
        flagged=set(bres.loc[bres["excess_significant"],"borough"])
        overlap=low_retention & flagged
        if overlap:
            print(f" {sorted(overlap)}")
else:
    print("ignore")

Excluded ['City of London']: 33 boroughs -> 32

Between-borough variance after removing sampling noise (tau^2): 0.001468
Implied between-borough SD of true residuals: 0.0383
Mean shrinkage weight: 0.705  (1.0 = no shrinkage, 0.0 = everything is noise)
Boroughs whose residual differs significantly from zero: 12 of 32

               borough   n  observed  expected  residual  residual_shrunk  excess_significant
  Barking and Dagenham 437    0.3630    0.2716    0.0915           0.0570                True
            Hillingdon 402    0.2852    0.1984    0.0868           0.0562                True
        Waltham Forest 450    0.3108    0.2241    0.0867           0.0552                True
             Redbridge 414    0.2699    0.2029    0.0670           0.0438                True
               Enfield 401    0.2850    0.2218    0.0632           0.0410                True
                Newham 405    0.3014    0.2423    0.0591           0.0380                True
              Hounslow 


NOTE: ['Enfield', 'Hillingdon'] show up in BOTH the low-retention list from
cell 2b and the significant-residual list above. Their residual score
rests on a thinner, less complete sample, so treat their ranking with
more caution than the others and say so if they appear in the write-up.


In [ ]:

if "borough" in test_df.columns:
    rank_corr=bres["observed"].corr(bres["residual_shrunk"],method="spearman")
    print(f"spearman correlation {rank_corr:.4f}")
  
    if abs(rank_corr) < 0.7:
        print(" different ranking")
    else:
        print("Rankings are similar")
       
    biggest_movers=bres.copy()
    biggest_movers["rank_observed"]=biggest_movers["observed"].rank(ascending=False)
    biggest_movers["rank_residual"]=biggest_movers["residual_shrunk"].rank(ascending=False)
    biggest_movers["rank_change"]=(biggest_movers["rank_observed"]- biggest_movers["rank_residual"])
    movers=biggest_movers.reindex(biggest_movers["rank_change"].abs().sort_values(ascending=False).index).head(8)
    print(movers[["borough","observed","expected","rank_observed","rank_residual","rank_change"]].round(4).to_string(index=False))

    OUT_B=(r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data" r"\borough_demand_residuals_v2.csv")
    bres.round(6).to_csv(OUT_B,index=False)
    print(f" {OUT_B}")

Spearman correlation, raw inactivity vs shrunk residual: 0.9600

Rankings are similar, so demographic adjustment mostly reorders at the
margins. Still worth reporting: it means the raw inactivity ranking wasn't
simply a demographic artefact, which is a useful thing to have shown.

Boroughs that move most when you adjust for demographics:
               borough  observed  expected  rank_observed  rank_residual  rank_change
                Merton    0.1951    0.1784           22.0           15.0          7.0
               Hackney    0.2218    0.2353           15.0           21.0         -6.0
           Westminster    0.2242    0.2133           13.0           17.0         -4.0
               Croydon    0.2199    0.1979           17.0           13.0          4.0
  Kingston upon Thames    0.1857    0.1833           23.0           19.0          4.0
Kensington and Chelsea    0.2004    0.2182           18.0           22.0         -4.0
             Redbridge    0.2699    0.2029            7.0 

In [ ]:
def subgroup_audit(d,y_true,y_prob,threshold,group_cols):
    out=[]
    for gc in group_cols:
        if gc not in d.columns:
            continue
        for level,idx in d.groupby(gc,observed=True).groups.items():
            pos=d.index.get_indexer(idx)
            yy,pp=y_true[pos],y_prob[pos]
            if len(yy) < 100 or len(np.unique(yy))<2:
                continue
            pred=(pp >= threshold).astype(int)
            tn,fp,fn,tp=confusion_matrix(yy,pred,labels=[0,1]).ravel()
            out.append({"attribute": gc,"group": str(level),"n": len(yy),
                "base_rate": yy.mean(),"auc": roc_auc_score(yy,pp),"mean_pred": pp.mean(),"calib_error": pp.mean()-yy.mean(),
                "fnr": fn / (fn+tp) if (fn+tp) else np.nan,"fpr": fp / (fp+tn) if (fp+tn) else np.nan,"selection_rate": pred.mean(),})
    return pd.DataFrame(out)

aud=test_df.reset_index(drop=True).copy()
aud["imd_tertile"]=pd.qcut(aud["imd_decile"],3,labels=["most deprived","middle","least deprived"])
THRESH=pick_threshold(train_df["inactive"].values,fitted_gb.predict_proba(train_df[DEMO_COLS])[:,1])

audit=subgroup_audit(aud,y_test,p_gb,THRESH,["disab3","gend3","age_band_collapsed","imd_tertile","nssec5"])
print(f"threshold {THRESH:.4f}")
print(audit.round(4).to_string(index=False))
for attr in audit["attribute"].unique():
    sub=audit[audit["attribute"] == attr]
    print(f"{attr:20s} auc spread {sub['auc'].max() - sub['auc'].min():.4f} | "f"fnr spread {sub['fnr'].max() - sub['fnr'].min():.4f} | " f"error {sub['calib_error'].abs().max():.4f}")


Deployment threshold (chosen on train): 0.2038

         attribute                                       group    n  base_rate    auc  mean_pred  calib_error    fnr    fpr  selection_rate
            disab3                         Limiting disability 2155     0.3573 0.6534     0.3183      -0.0390 0.0818 0.7978          0.8408
            disab3                               No disability 9978     0.1907 0.6210     0.1785      -0.0122 0.6174 0.2041          0.2381
            disab3                     Non-limiting disability 2064     0.1555 0.6396     0.1539      -0.0017 0.6293 0.1354          0.1720
             gend3                                      Female 7944     0.2244 0.6559     0.2035      -0.0210 0.4823 0.2844          0.3367
             gend3                                        Male 6184     0.1932 0.6645     0.1865      -0.0067 0.4795 0.2421          0.2959
age_band_collapsed                                       16-24 1262     0.2092 0.6384     0.1911      -0.0181 0.

In [ ]:
#checking if threshold choice was good

target_fnr=audit["fnr"].median()
print(f"median fnr {target_fnr:.4f}")
rows=[]
for attr in ["disab3","gend3","imd_tertile"]:
    if attr not in aud.columns:
        continue
    for level,idx in aud.groupby(attr,observed=True).groups.items():
        pos=aud.index.get_indexer(idx)
        yy,pp=y_test[pos],p_gb[pos]
        if len(yy) < 100 or yy.sum() < 20:
            continue
        # threshold at which this group's fnr equals the target
        cand=np.quantile(pp[yy == 1],target_fnr)
        pred=(pp >= THRESH).astype(int)
        tn,fp,fn,tp=confusion_matrix(yy,pred,labels=[0,1]).ravel()
        rows.append({"attribute": attr,"group": str(level),"n": len(yy),
                     "fnr_at_global_thresh": fn / (fn+tp),"thresh_for_equal_fnr": cand,"shift_needed": cand-THRESH})
eq=pd.DataFrame(rows)
print(eq.round(4).to_string(index=False))


Median FNR across all subgroups: 0.4759

  attribute                   group    n  fnr_at_global_thresh  thresh_for_equal_fnr  shift_needed
     disab3     Limiting disability 2155                0.0818                0.3439        0.1402
     disab3           No disability 9978                0.6174                0.1691       -0.0347
     disab3 Non-limiting disability 2064                0.6293                0.1551       -0.0486
      gend3                  Female 7944                0.4823                0.2026       -0.0011
      gend3                    Male 6184                0.4795                0.1991       -0.0047
imd_tertile           most deprived 6084                0.3376                0.2477        0.0440
imd_tertile                  middle 4740                0.5412                0.1728       -0.0310
imd_tertile          least deprived 3373                0.7935                0.1460       -0.0578

A shift_needed near zero means the global threshold already treats 

In [ ]:
# checks auc within each group

overall_auc=roc_auc_score(y_test,p_gb)
print(f" test auc {overall_auc:.4f}")
rows=[]
for attr in ["nssec5","disab3","imd_tertile","age_band_collapsed"]:
    if attr not in aud.columns:
        continue
    sub=audit[audit["attribute"] == attr]
    if not len(sub):
        continue
    w=sub["n"] / sub["n"].sum()
    rows.append({
        "held_constant": attr,"n_groups": len(sub),
        "weighted_within_group_auc": float((sub["auc"] * w).sum()),"min_group_auc": sub["auc"].min(),"max_group_auc": sub["auc"].max(),})
wg=pd.DataFrame(rows)
wg["retained_vs_overall"]=(wg["weighted_within_group_auc"]-0.5) / (overall_auc-0.5)
print(wg.round(4).to_string(index=False))

worst=wg.loc[wg["retained_vs_overall"].idxmin()]
print(f" {worst['held_constant']}  "
      f"{worst['retained_vs_overall']:.0%} of the discrimination.")



Overall test AUC: 0.6617

     held_constant  n_groups  weighted_within_group_auc  min_group_auc  max_group_auc  retained_vs_overall
            nssec5         4                     0.6033         0.5757         0.6596               0.6385
            disab3         3                     0.6286         0.6210         0.6534               0.7951
       imd_tertile         3                     0.6435         0.5724         0.6779               0.8872
age_band_collapsed         6                     0.6572         0.6352         0.6933               0.9720

retained_vs_overall is the share of the model's discriminative power that
survives when that variable is held constant. 1.0 means the variable
contributes nothing unique; 0.0 means the model is ONLY that variable.

Weakest case: holding nssec5 constant retains 64% of the discrimination.
Anything comfortably above roughly 60% means the model ranks individuals
within groups, not just between them, and the lopsided selection rates in
the

In [ ]:


def net_benefit(y,p,pt):
    pred=(p>=pt).astype(int)
    tp=((pred==1)&(y == 1)).sum()
    fp=((pred==1)&(y==0)).sum()
    n=len(y)
    return tp/n-(fp/n)*(pt/(1-pt))

thresholds=np.arange(0.05,0.61,0.025)
imd_only=Pipeline([("prep",build_preprocessor([],["imd_decile"])),("clf",LogisticRegression(max_iter=2000,random_state=RANDOM_STATE))])
imd_only.fit(train_df[["imd_decile"]],train_df["inactive"])
p_imd=imd_only.predict_proba(test_df[["imd_decile"]])[:,1]

dca=[]
for pt in thresholds:
    dca.append({"threshold": pt,"nb_model": net_benefit(y_test,p_gb,pt),"nb_imd_only": net_benefit(y_test,p_imd,pt),
        "nb_treat_all": prevalence-(1-prevalence) * (pt / (1-pt)),"nb_treat_none": 0.0,})
dca=pd.DataFrame(dca)
dca["model_best"]=((dca["nb_model"] > dca["nb_treat_all"]) &(dca["nb_model"] > dca["nb_treat_none"]) &(dca["nb_model"] > dca["nb_imd_only"]))
print(dca.round(4).to_string(index=False))

useful=dca[dca["model_best"]]
if len(useful):
    print(f"{useful['threshold'].min():.3f} to {useful['threshold'].max():.3f}.")

else:
    print("does not dominate")


 threshold  nb_model  nb_imd_only  nb_treat_all  nb_treat_none  model_best
     0.050    0.1694       0.1694        0.1694            0.0       False
     0.075    0.1469       0.1469        0.1469            0.0       False
     0.100    0.1224       0.1232        0.1232            0.0       False
     0.125    0.0987       0.0982        0.0982            0.0        True
     0.150    0.0800       0.0699        0.0716            0.0        True
     0.175    0.0674       0.0492        0.0435            0.0        True
     0.200    0.0570       0.0316        0.0136            0.0        True
     0.225    0.0472       0.0203       -0.0182            0.0        True
     0.250    0.0385       0.0025       -0.0521            0.0        True
     0.275    0.0321       0.0000       -0.0884            0.0        True
     0.300    0.0257       0.0000       -0.1273            0.0        True
     0.325    0.0219       0.0000       -0.1691            0.0        True
     0.350    0.0156     

In [ ]:
#main table

order=np.argsort(-p_gb)
y_sorted=y_test[order]
total_inactive=y_sorted.sum()

cap_rows=[]
for pct in [0.05,0.10,0.15,0.20,0.25,0.30,0.40,0.50]:
    k=int(len(y_sorted) * pct)
    reached=y_sorted[:k]
    cap_rows.append({"reach_pct": pct,"people_reached": k,"inactive_reached": int(reached.sum()), "pct_of_inactive_captured": reached.sum() / total_inactive,"precision": reached.mean(),"lift_vs_random": (reached.sum() / total_inactive) / pct,})
cap=pd.DataFrame(cap_rows)
print(f"test year {len(y_sorted):,} adults, {int(total_inactive):,} inactive "f"({prevalence:.1%})")
print(cap.round(4).to_string(index=False))
best=cap.loc[cap["lift_vs_random"].idxmax()]
print(f"efficiency is highest  at {best['reach_pct']:.0%}  "f"{best['lift_vs_random']:.2f} ")

OUT_C=(r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data"r"\targeting_capacity_v2.csv")
cap.round(6).to_csv(OUT_C,index=False)
print(f"\n {OUT_C}")

Test year: 14,197 adults, 2,994 inactive (21.1%)

 reach_pct  people_reached  inactive_reached  pct_of_inactive_captured  precision  lift_vs_random
      0.05             709               358                    0.1196     0.5049          2.3914
      0.10            1419               661                    0.2208     0.4658          2.2077
      0.15            2129               910                    0.3039     0.4274          2.0263
      0.20            2839              1122                    0.3747     0.3952          1.8737
      0.25            3549              1305                    0.4359     0.3677          1.7435
      0.30            4259              1485                    0.4960     0.3487          1.6533
      0.40            5678              1784                    0.5959     0.3142          1.4896
      0.50            7098              2024                    0.6760     0.2852          1.3520

Efficiency peaks at 5% reach: 2.39x better than random selection.
L

In [ ]:
#model card

card={"train_n": len(train_df),"test_n": len(test_df),"train_years": f"{TRAIN_YEARS[0]} to {TRAIN_YEARS[-1]}",
    "test_year": TEST_YEARS[0],"features": ", ".join(DEMO_COLS),"test_auc": roc_auc_score(y_test,p_gb),"test_ap": average_precision_score(y_test,p_gb),
    "ap_floor": prevalence,"brier": brier_score_loss(y_test,p_gb),"brier_base_rate": brier_score_loss(y_test,np.full_like(p_gb,prevalence)),"rolling_auc_mean": roll_gb["test_auc"].mean(),
    "rolling_auc_sd": roll_gb["test_auc"].std(),"age_only_baseline_auc": BASELINE_AUC,"deployment_threshold": THRESH,"capture_at_20pct_reach": cap.loc[cap["reach_pct"] == 0.20,"pct_of_inactive_captured"].iloc[0],
    "max_fnr_spread": audit.groupby("attribute")["fnr"].apply(lambda s: s.max()-s.min()).max(),"complete_case_rate": len(m1) / len(df),}
for k,v in card.items():
    print(f"{k:28s} {v:.4f}" if isinstance(v,float) else f"{k:28s} {v}")

OUT_D=(r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data"r"\model_card_v2.csv")
pd.DataFrame([card]).to_csv(OUT_D,index=False)
print(f"\n {OUT_D}")

train_n                      88204
test_n                       14197
train_years                  2016-17 to 2021-22
test_year                    2022-23
features                     age_band_collapsed, imd_decile, disab3, nssec5, gend3
test_auc                     0.6617
test_ap                      0.3628
ap_floor                     0.2109
brier                        0.1556
brier_base_rate              0.1664
rolling_auc_mean             0.6504
rolling_auc_sd               0.0091
age_only_baseline_auc        0.5359
deployment_threshold         0.2038
capture_at_20pct_reach       0.3747
max_fnr_spread               0.8284
complete_case_rate           0.8702

Saved to C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\model_card_v2.csv
